# 🧠 Salient Object Detection — End-to-End ML Project
> **Cohort V | Project #3**  
> Full pipeline: data → model → train → evaluate → demo

---
### How to use this notebook
Run cells **top to bottom**. Each section is clearly labelled.  
A GPU runtime is strongly recommended: `Runtime → Change runtime type → T4 GPU`

## ⚙️ 0. Setup — Install dependencies & mount Drive

In [ ]:
# Mount Google Drive (optional — for saving checkpoints persistently)
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted ✓')

In [ ]:
# Install required libraries
!pip install -q albumentations gradio tqdm opencv-python-headless

In [ ]:
# Upload and extract the project ZIP
from google.colab import files
import zipfile, os

print('Upload sod_project.zip ...')
uploaded = files.upload()   # choose sod_project.zip

with zipfile.ZipFile('sod_project.zip', 'r') as z:
    z.extractall('/content/')

os.chdir('/content/sod_project')
print('Working directory:', os.getcwd())
!ls

## 📦 1. Download & Prepare Dataset (DUTS)

In [ ]:
# ── Option A: Download DUTS-TE (smaller test set, ~5k images) ──────────────
# For a quick run use ECSSD (~1k images) — uncomment the block you want

import os, zipfile, shutil
from pathlib import Path

# ---- ECSSD (recommended for quick testing, ~1000 images) ----
# Download from: https://www.cse.cuhk.edu.hk/leojia/projects/hsaliency/dataset.html
# Or use the Kaggle version:
!pip install -q kaggle

# Upload your kaggle.json first
from google.colab import files
print('Upload kaggle.json (from https://www.kaggle.com/account):')
uploaded = files.upload()

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Download ECSSD dataset from Kaggle
!kaggle datasets download -d rahidul1/ecssd-salient-object-detection-dataset -p /content/raw_data --unzip
print('Download complete ✓')

In [ ]:
# ── Organize into dataset/images and dataset/masks structure ──────────────
import shutil
from pathlib import Path

RAW = Path('/content/raw_data')
DST = Path('/content/sod_project/dataset')
(DST / 'images').mkdir(parents=True, exist_ok=True)
(DST / 'masks').mkdir(parents=True, exist_ok=True)

# Adjust these glob patterns to match your downloaded dataset layout
# Common ECSSD layout: images/*.jpg  masks/*.png
img_files  = list(RAW.rglob('*.jpg')) + list(RAW.rglob('*.png'))
mask_files = []

for f in sorted(RAW.rglob('*')):
    name = f.name.lower()
    # Masks are typically in a folder named 'ground_truth_mask' or 'masks'
    if 'mask' in str(f.parent).lower() or 'ground' in str(f.parent).lower():
        if f.suffix in ['.png', '.jpg']:
            shutil.copy(f, DST / 'masks' / f.name)
    elif f.suffix in ['.jpg', '.jpeg']:
        if 'mask' not in name and 'ground' not in name:
            shutil.copy(f, DST / 'images' / f.name)

n_imgs  = len(list((DST/'images').glob('*')))
n_masks = len(list((DST/'masks').glob('*')))
print(f'Organized: {n_imgs} images, {n_masks} masks')

In [ ]:
# ── Verify dataset with a quick visual check ──────────────────────────────
import sys
sys.path.insert(0, '/content/sod_project')
from data_loader import get_dataloaders, visualize_batch

train_loader, val_loader, test_loader = get_dataloaders(
    '/content/sod_project/dataset',
    image_size=224, batch_size=8
)
visualize_batch(train_loader, n=4)
print('Dataset looks good ✓')

## 🏗️ 2. Model Architecture Sanity Check

In [ ]:
import torch
from sod_model import SODNet, SODNetPlus, count_parameters

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}\n')

print('── Baseline SODNet ──')
baseline = SODNet().to(device)
count_parameters(baseline)
x = torch.randn(2, 3, 224, 224).to(device)
print('Output shape:', baseline(x).shape)

print('\n── Improved SODNetPlus ──')
improved = SODNetPlus().to(device)
count_parameters(improved)
print('Output shape:', improved(x).shape)

## 🚀 3. Train — Baseline Model

In [ ]:
import json
from train import train, DEFAULT_CONFIG

cfg_baseline = DEFAULT_CONFIG.copy()
cfg_baseline.update({
    'data_dir'       : '/content/sod_project/dataset',
    'model_type'     : 'baseline',
    'epochs'         : 20,
    'batch_size'     : 16,
    'lr'             : 1e-3,
    'checkpoint_dir' : '/content/sod_project/checkpoints',
    'log_file'       : '/content/sod_project/training_log_baseline.json',
})

model_baseline, history_baseline = train(cfg_baseline)

# Copy best checkpoint with model name
import shutil
shutil.copy('checkpoints/best.pt', 'checkpoints/best_baseline.pt')
print('Baseline training complete ✓')

## 🚀 4. Train — Improved Model (Experiment)

In [ ]:
from train import train, DEFAULT_CONFIG
import shutil

# Remove previous latest checkpoint so we start fresh
import os
if os.path.exists('checkpoints/latest.pt'):
    os.remove('checkpoints/latest.pt')

cfg_improved = DEFAULT_CONFIG.copy()
cfg_improved.update({
    'data_dir'       : '/content/sod_project/dataset',
    'model_type'     : 'improved',
    'epochs'         : 20,
    'batch_size'     : 16,
    'lr'             : 1e-3,
    'checkpoint_dir' : '/content/sod_project/checkpoints',
    'log_file'       : '/content/sod_project/training_log_improved.json',
})

model_improved, history_improved = train(cfg_improved)
shutil.copy('checkpoints/best.pt', 'checkpoints/best_improved.pt')
print('Improved training complete ✓')

## 📊 5. Evaluate — Baseline vs Improved

In [ ]:
from evaluate import evaluate
import json

print('\n' + '='*50)
print('  BASELINE MODEL EVALUATION')
print('='*50)
metrics_baseline = evaluate(
    data_dir          = '/content/sod_project/dataset',
    checkpoint_path   = 'checkpoints/best_baseline.pt',
    model_type        = 'baseline',
    output_dir        = 'outputs/baseline'
)

print('\n' + '='*50)
print('  IMPROVED MODEL EVALUATION')
print('='*50)
metrics_improved = evaluate(
    data_dir          = '/content/sod_project/dataset',
    checkpoint_path   = 'checkpoints/best_improved.pt',
    model_type        = 'improved',
    output_dir        = 'outputs/improved'
)

In [ ]:
# ── Comparison Table ──────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame([
    {'Model': 'Baseline (SODNet)',  **metrics_baseline},
    {'Model': 'Improved (SODNetPlus)', **metrics_improved},
]).set_index('Model').round(4)

print('\n── Comparison Table ──')
print(df.to_string())

# Style for display
df.style.background_gradient(cmap='RdYlGn', axis=0)

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────
from evaluate import plot_training_history
import matplotlib.pyplot as plt

plot_training_history(
    log_file='training_log_baseline.json',
    save_path='outputs/baseline/training_curves.png'
)

## 🎨 6. Visualize Predictions

In [ ]:
from evaluate import visualize_predictions
from sod_model import SODNet, SODNetPlus
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load improved model
model = SODNetPlus().to(device)
ckpt  = torch.load('checkpoints/best_improved.pt', map_location=device)
model.load_state_dict(ckpt['model_state'])

visualize_predictions(
    model, test_loader, device,
    n_samples=8,
    save_path='outputs/improved/predictions.png'
)

## 🖥️ 7. Live Demo — Single Image Inference

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from google.colab import files
from sod_model import SODNetPlus

# Upload an image
print('Upload any image to run inference:')
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

# Load & preprocess
image = cv2.imread(img_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
orig_h, orig_w = image.shape[:2]

transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])
tensor = transform(image=image)['image'].unsqueeze(0).to(device)

# Inference
model.eval()
t0 = time.perf_counter()
with torch.no_grad():
    pred = model(tensor).squeeze().cpu().numpy()
ms = (time.perf_counter() - t0) * 1000

# Post-process
pred_r    = cv2.resize(pred, (orig_w, orig_h))
mask_bin  = (pred_r > 0.5).astype(np.uint8)
heatmap   = cv2.applyColorMap((pred_r*255).astype(np.uint8), cv2.COLORMAP_HOT)
heatmap   = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
overlay   = image.copy().astype(np.float32)
overlay[mask_bin==1] = overlay[mask_bin==1]*0.5 + np.array([255,60,60])*0.5
overlay   = overlay.clip(0,255).astype(np.uint8)

# Display
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
data  = [image, heatmap, (pred_r*255).astype(np.uint8), overlay]
titles = ['Input Image', 'Saliency Heatmap', 'Predicted Mask', 'Overlay']
cmaps  = [None, None, 'gray', None]
for ax, d, t, c in zip(axes, data, titles, cmaps):
    ax.imshow(d, cmap=c); ax.set_title(t, fontweight='bold'); ax.axis('off')

plt.suptitle(f'Inference time: {ms:.1f} ms  |  Device: {device}',
             fontsize=12, color='gray')
plt.tight_layout()
plt.savefig('demo_result.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Inference time: {ms:.1f} ms')

## 🌐 8. Launch Gradio Web App Demo

In [ ]:
# Make sure both checkpoints exist before launching
import os
import shutil

# Point both to best available checkpoint
if not os.path.exists('checkpoints/best_baseline.pt') and os.path.exists('checkpoints/best.pt'):
    shutil.copy('checkpoints/best.pt', 'checkpoints/best_baseline.pt')
if not os.path.exists('checkpoints/best_improved.pt') and os.path.exists('checkpoints/best.pt'):
    shutil.copy('checkpoints/best.pt', 'checkpoints/best_improved.pt')

# Launch Gradio (share=True creates a public URL)
!python app.py

## 💾 9. Save Everything to Google Drive
Run this after training to keep your work safe.

In [ ]:
import shutil

DRIVE_DEST = '/content/drive/MyDrive/SOD_Project'
shutil.copytree('/content/sod_project', DRIVE_DEST, dirs_exist_ok=True)
print(f'Saved to Google Drive → {DRIVE_DEST}')